# MMC Semantic Search Pipeline for Google Colab

This notebook provides a complete end-to-end semantic search pipeline for MMC articles:
- **Data Extraction & Cleaning**: Convert YAML to cleaned JSON with text preprocessing
- **Embedding Generation**: Create semantic embeddings using multilingual sentence transformers
- **Index Building**: Construct FAISS indices for fast similarity search
- **Search & Retrieval**: Execute cosine similarity queries over article database
- **Visualization**: Interactive UMAP 2D projections of article clusters
- **Quality Analysis**: Silhouette analysis comparing KMeans clusters vs article topics

**Best used with Google Colab GPU runtime for fast embedding generation.**

## Section 1: Setup and Installation

In [ ]:
import os
import sys
from pathlib import Path

# Check if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✓ Running in Google Colab")
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = Path('/content/drive/MyDrive/mmc_projekt')
else:
    print("✓ Running locally")
    BASE_DIR = Path.cwd()

# Ensure data directory exists
(BASE_DIR / 'data').mkdir(parents=True, exist_ok=True)
print(f"Base directory: {BASE_DIR}")

In [ ]:
# Install required dependencies
import subprocess

packages = [
    "numpy",
    "scipy",
    "sentence-transformers",
    "torch",
    "faiss-cpu",
    "scikit-learn",
    "umap-learn",
    "plotly",
    "pyyaml",
]

print("Installing dependencies...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        print(f"✓ {package}")
    except subprocess.CalledProcessError:
        print(f"✗ Failed to install {package}")

print("\nAll dependencies installed!")

In [ ]:
# Import all required libraries
import argparse
import json
import logging
import re
import unicodedata
import hashlib
from pathlib import Path
from typing import Any
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Lock
from urllib.parse import parse_qs, urlparse

import numpy as np
import yaml
import torch
from sentence_transformers import SentenceTransformer, CrossEncoders
import faiss
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import normalize
import umap
import plotly.colors as plotly_colors
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

print("✓ All imports successful!")

## Utility Functions

In [ ]:
# Text cleaning functions
YAML_LOADER = getattr(yaml, "CSafeLoader", yaml.SafeLoader)

def fix_mojibake(text: str) -> str:
    """Fix CP1250 encoding issues (e.g., priĹˇel -> prišel)"""
    try:
        return text.encode("cp1250").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text

def clean_text_value(value: str) -> str:
    """Clean text while preserving natural language structure."""
    value = fix_mojibake(value)
    value = re.sub(r"<[^>]+>", "", value)  # Remove HTML tags
    value = re.sub(r"http\S+|www\.\S+", "", value)  # Remove URLs
    value = unicodedata.normalize("NFKC", value)  # Normalize unicode
    value = value.replace("\u00a0", " ")  # Replace non-breaking spaces
    value = re.sub(r"[ \t\r\f\v]+", " ", value)  # Normalize whitespace
    value = re.sub(r"\n\s*\n+", "\n\n", value)
    return value.strip()

def clean_value(value: Any, field_name: str | None = None) -> Any:
    """Recursively clean values while preserving structure."""
    if isinstance(value, str):
        if field_name == "url":
            return value
        return clean_text_value(value)
    if isinstance(value, list):
        cleaned_items = [clean_value(item, field_name) for item in value]
        return [item for item in cleaned_items if item not in (None, "", [], {})]
    if isinstance(value, dict):
        return {key: clean_value(item, key) for key, item in value.items()}
    return value

def normalize_keywords(*values: Any) -> list[str]:
    """Normalize keyword-like fields into a deduplicated list."""
    keywords: list[str] = []
    seen: set[str] = set()
    for value in values:
        if isinstance(value, str):
            candidates = [item.strip() for item in re.split(r"[,;]", value)]
        elif isinstance(value, list):
            candidates = [item for item in value if isinstance(item, str)]
        else:
            continue
        for candidate in candidates:
            keyword = clean_text_value(candidate)
            if not keyword:
                continue
            normalized = keyword.casefold()
            if normalized in seen:
                continue
            seen.add(normalized)
            keywords.append(keyword)
    return keywords

print("✓ Text cleaning functions loaded")

## Section 2: Load Articles and Prepare Data

In [ ]:
def build_embedding_text(article: dict[str, Any]) -> str:
    """Build high-quality text representation for embedding."""
    parts: list[str] = []
    
    title = article.get("title")
    if isinstance(title, str) and title:
        parts.append(title)
    
    keywords = normalize_keywords(article.get("keywords"), article.get("gpt_keywords"))
    if keywords:
        parts.append("Ključne besede: " + ", ".join(keywords))
    
    lead = article.get("lead")
    if isinstance(lead, str) and lead:
        parts.append(lead)
    
    paragraphs = article.get("paragraphs", [])
    if isinstance(paragraphs, list):
        for paragraph in paragraphs[:3]:
            if isinstance(paragraph, str) and paragraph:
                parts.append(paragraph)
    
    text = "\n\n".join(parts)
    return clean_text_value(text)

def extract_yaml_to_cleaned_json(input_path: Path, sample_size: int = None) -> list[dict[str, Any]]:
    """Extract YAML data, clean it, and return as JSON records."""
    print(f"Loading YAML from: {input_path}")
    
    with input_path.open("r", encoding="utf-8") as source_file:
        data = yaml.load(source_file, Loader=YAML_LOADER)
    
    if not isinstance(data, list):
        raise ValueError("Expected YAML file to contain a list of articles.")
    
    if sample_size:
        data = data[:sample_size]
        print(f"Using sample of {sample_size} articles")
    
    cleaned_records = []
    
    for i, record in enumerate(data):
        if not isinstance(record, dict):
            continue
        
        cleaned = clean_value(record)
        keywords = normalize_keywords(cleaned.get("keywords"), cleaned.get("gpt_keywords"))
        
        embedding_text = build_embedding_text(cleaned)
        if not embedding_text:
            continue
        
        processed = {
            "id": cleaned.get("id", i),
            "title": cleaned.get("title", ""),
            "text": embedding_text,
            "keywords": keywords,
            "category": cleaned.get("category") or cleaned.get("topics"),
            "date": cleaned.get("date"),
            "url": cleaned.get("url"),
        }
        
        cleaned_records.append(processed)
        
        if (i + 1) % 5000 == 0:
            print(f"Processed {i + 1} articles...")
    
    print(f"Total cleaned articles: {len(cleaned_records)}")
    return cleaned_records

# Example: Load YAML data (adjust path based on your setup)
yaml_path = BASE_DIR / "data" / "mmc.yaml" / "mmc.yaml"
sample_size = 1000  # Start with 1000 for testing, use None for all articles

print(f"Looking for YAML at: {yaml_path}")
print(f"File exists: {yaml_path.exists()}")

if yaml_path.exists():
    cleaned_articles = extract_yaml_to_cleaned_json(yaml_path, sample_size=sample_size)
    print(f"✓ Loaded and cleaned {len(cleaned_articles)} articles")
else:
    print("⚠ YAML file not found. Please upload your data or adjust the path.")

## Section 3: Generate Article Embeddings

In [ ]:
def build_weighted_embedding_text(article: dict[str, Any]) -> str:
    """Build weighted text with title repetition for stronger signal."""
    title = str(article.get("title", "")).strip()
    body_text = str(article.get("text", "")).strip()
    
    parts: list[str] = []
    if title:
        parts.append(f"title: {title}")
        parts.append(f"title: {title}")
        parts.append(f"title: {title}")
    if body_text:
        parts.append(f"passage: {body_text}")
    
    return " ".join(parts).strip()

def resolve_device(device_choice: str = "auto") -> str:
    """Resolve to appropriate computing device."""
    if device_choice != "auto":
        return device_choice
    
    if torch.cuda.is_available():
        return "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

def embed_articles_batch(
    articles: list[dict[str, Any]],
    model_name: str = "intfloat/multilingual-e5-large",
    batch_size: int = 64,
    device: str = "auto",
) -> np.ndarray:
    """Generate embeddings for articles."""
    device = resolve_device(device)
    print(f"Using device: {device}")
    
    print(f"Loading model: {model_name}")
    model = SentenceTransformer(model_name, device=device)
    
    texts = [build_weighted_embedding_text(article) for article in articles]
    
    print(f"Encoding {len(texts)} articles (batch_size={batch_size}, normalize=True)")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    
    embeddings = embeddings.astype(np.float32, copy=False)
    print(f"✓ Embedding complete. Shape: {embeddings.shape}")
    return embeddings

# Generate embeddings
if yaml_path.exists() and 'cleaned_articles' in globals():
    embeddings = embed_articles_batch(cleaned_articles, batch_size=32)
    print(f"Generated embeddings: {embeddings.shape[0]} articles, {embeddings.shape[1]} dimensions")

## Section 4: Save and Organize Outputs

In [ ]:
def save_embeddings_and_metadata(
    embeddings: np.ndarray,
    articles: list[dict[str, Any]],
    output_dir: Path,
    model_name: str = "intfloat/multilingual-e5-large",
) -> None:
    """Save embeddings, metadata, and configuration."""
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Saving outputs to: {output_dir}")
    
    # Save embeddings
    embeddings_path = output_dir / "embeddings.npy"
    np.save(embeddings_path, embeddings)
    print(f"✓ Saved embeddings: {embeddings_path}")
    
    # Save metadata
    metadata_path = output_dir / "metadata.jsonl"
    with metadata_path.open("w", encoding="utf-8") as meta_file:
        for article in articles:
            metadata = {
                "id": article["id"],
                "title": article["title"],
                "url": article["url"],
                "date": article["date"],
                "category": article["category"],
                "keywords": article["keywords"],
                "text": article["text"],
            }
            meta_file.write(json.dumps(metadata, ensure_ascii=False) + "\n")
    print(f"✓ Saved metadata: {metadata_path}")
    
    # Save config
    config_path = output_dir / "index_config.json"
    config = {
        "model_name": model_name,
        "normalize_embeddings": True,
        "metric": "cosine",
        "text_weighting": "title repeated 3x before passage body",
        "num_vectors": int(embeddings.shape[0]),
        "vector_dim": int(embeddings.shape[1]),
        "embeddings_file": "embeddings.npy",
        "metadata_file": "metadata.jsonl",
    }
    with config_path.open("w", encoding="utf-8") as cfg_file:
        json.dump(config, cfg_file, ensure_ascii=False, indent=2)
    print(f"✓ Saved config: {config_path}")

# Save outputs
if 'embeddings' in globals() and 'cleaned_articles' in globals():
    output_dir = BASE_DIR / "data" / "mmc_embeddings"
    save_embeddings_and_metadata(embeddings, cleaned_articles, output_dir)

## Section 5: Build and Execute FAISS Search Index

In [ ]:
def load_embeddings_and_metadata(embeddings_dir: Path) -> tuple[np.ndarray, list[dict[str, Any]]]:
    """Load embeddings and metadata."""
    embeddings_path = embeddings_dir / "embeddings.npy"
    metadata_path = embeddings_dir / "metadata.jsonl"
    
    embeddings = np.load(embeddings_path)
    
    metadata = []
    with metadata_path.open("r", encoding="utf-8") as meta_file:
        for line in meta_file:
            if line.strip():
                metadata.append(json.loads(line))
    
    return embeddings, metadata

def build_faiss_index(embeddings: np.ndarray, metric: str = "cosine") -> faiss.Index:
    """Build FAISS index."""
    d = embeddings.shape[1]
    
    if metric == "cosine":
        normalized_embeddings = normalize(embeddings, norm="l2")
        index = faiss.IndexFlatIP(d)
        index.add(normalized_embeddings.astype(np.float32))
    else:
        index = faiss.IndexFlatL2(d)
        index.add(embeddings.astype(np.float32))
    
    print(f"✓ Built FAISS index: {index.ntotal} vectors, metric={metric}")
    return index

def search_faiss(
    query_text: str,
    index: faiss.Index,
    metadata: list[dict[str, Any]],
    embeddings: np.ndarray,
    model: SentenceTransformer,
    k: int = 5,
    metric: str = "cosine",
) -> list[dict[str, Any]]:
    """Execute FAISS search."""
    query_embedding = model.encode([query_text], normalize_embeddings=True).astype(np.float32)
    
    if metric == "cosine":
        query_embedding = normalize(query_embedding, norm="l2")
    
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for idx, distance in zip(indices[0], distances[0]):
        if idx >= 0:
            article = metadata[int(idx)]
            results.append({
                "rank": len(results) + 1,
                "score": float(distance),
                "id": article["id"],
                "title": article["title"],
                "url": article["url"],
                "date": article["date"],
                "category": article["category"],
                "snippet": article["text"][:200] + "...",
            })
    
    return results

# Load embeddings and build index
embeddings_dir = BASE_DIR / "data" / "mmc_embeddings"
if embeddings_dir.exists():
    embeddings, metadata = load_embeddings_and_metadata(embeddings_dir)
    faiss_index = build_faiss_index(embeddings, metric="cosine")
    model_embedding = SentenceTransformer("intfloat/multilingual-e5-large")
    print(f"✓ Ready for search: {len(metadata)} articles indexed")

## Section 6: Perform Similarity Search Queries

In [ ]:
# Example search queries
if embeddings_dir.exists() and 'faiss_index' in globals():
    test_queries = [
        "artificial intelligence and machine learning",
        "climate change and environment",
        "sports and Olympics",
    ]
    
    for query in test_queries:
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print('='*60)
        
        results = search_faiss(
            query,
            faiss_index,
            metadata,
            embeddings,
            model_embedding,
            k=5,
        )
        
        for result in results:
            print(f"\n[{result['rank']}] Score: {result['score']:.4f}")
            print(f"    Title: {result['title']}")
            print(f"    Category: {result['category']}")
            print(f"    URL: {result['url']}")
            print(f"    Snippet: {result['snippet']}")

## Section 7: Visualize UMAP Dimensionality Reduction

In [ ]:
def compute_kmeans_and_umap(
    embeddings: np.ndarray,
    n_clusters: int = 10,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute KMeans clusters and UMAP 2D projection."""
    print(f"Computing KMeans with {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    
    print(f"Computing UMAP projection...")
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="cosine",
        n_components=2,
        random_state=random_state,
    )
    umap_2d = reducer.fit_transform(embeddings)
    
    return labels, umap_2d

def create_interactive_umap_plot(
    umap_2d: np.ndarray,
    kmeans_labels: np.ndarray,
    topic_labels: np.ndarray,
    metadata: list[dict[str, Any]],
) -> go.Figure:
    """Create interactive Plotly visualization."""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("KMeans Clusters (10)", "Article Topics"),
        specs=[[{"type": "scatter"}, {"type": "scatter"}]],
    )
    
    # KMeans plot (left)
    colors_km = plotly_colors.qualitative.Plotly + plotly_colors.qualitative.Light24
    unique_clusters = np.unique(kmeans_labels)
    for cluster_id in sorted(unique_clusters):
        mask = kmeans_labels == cluster_id
        indices = np.where(mask)[0]
        
        hover_texts = []
        for idx in indices:
            article = metadata[idx] if idx < len(metadata) else {}
            hover_texts.append(f"<b>{article.get('title', 'N/A')}</b><br>Cluster: {cluster_id}")
        
        fig.add_trace(
            go.Scatter(
                x=umap_2d[mask, 0],
                y=umap_2d[mask, 1],
                mode="markers",
                marker=dict(size=6, color=colors_km[cluster_id % len(colors_km)]),
                name=f"Cluster {cluster_id}",
                text=hover_texts,
                hovertemplate="%{text}<extra></extra>",
            ),
            row=1, col=1,
        )
    
    # Topics plot (right)
    unique_topics = np.unique(topic_labels)
    for topic_id in sorted(unique_topics):
        mask = topic_labels == topic_id
        
        hover_texts = []
        for idx, is_topic in enumerate(mask):
            if is_topic and idx < len(metadata):
                article = metadata[idx]
                hover_texts.append(f"<b>{article.get('title', 'N/A')}</b><br>Category: {article.get('category', 'N/A')}")
        
        if np.any(mask):
            fig.add_trace(
                go.Scatter(
                    x=umap_2d[mask, 0],
                    y=umap_2d[mask, 1],
                    mode="markers",
                    marker=dict(size=6, color=colors_km[topic_id % len(colors_km)]),
                    name=f"Topic {topic_id}",
                    text=hover_texts,
                    hovertemplate="%{text}<extra></extra>",
                ),
                row=1, col=2,
            )
    
    fig.update_layout(
        title_text="UMAP Projection: Semantic Clusters vs. Article Topics",
        height=500,
        width=1200,
        hovermode="closest",
    )
    
    return fig

# Compute UMAP projection if embeddings are available
if embeddings_dir.exists() and 'embeddings' in globals():
    kmeans_labels, umap_2d = compute_kmeans_and_umap(embeddings, n_clusters=10)
    
    # Create topic labels from metadata
    unique_categories = {}
    topic_labels = np.zeros(len(metadata), dtype=int)
    category_id = 0
    
    for idx, article in enumerate(metadata):
        category = article.get("category", "uncategorized")
        if category not in unique_categories:
            unique_categories[category] = category_id
            category_id += 1
        topic_labels[idx] = unique_categories[category]
    
    # Create visualization
    fig = create_interactive_umap_plot(umap_2d, kmeans_labels, topic_labels, metadata)
    fig.show()

## Section 8: Calculate Silhouette Analysis

In [ ]:
def compute_silhouette_analysis(
    embeddings: np.ndarray,
    labels: np.ndarray,
    label_name: str = "KMeans",
) -> dict[str, Any]:
    """Compute silhouette metrics."""
    sil_scores = silhouette_samples(embeddings, labels, metric="cosine")
    sil_mean = silhouette_score(embeddings, labels, metric="cosine")
    
    results = {
        "label_name": label_name,
        "mean_score": sil_mean,
        "all_scores": sil_scores.tolist(),
        "per_cluster": {},
    }
    
    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        cluster_scores = sil_scores[mask]
        results["per_cluster"][int(cluster_id)] = {
            "mean": float(np.mean(cluster_scores)),
            "median": float(np.median(cluster_scores)),
            "std": float(np.std(cluster_scores)),
            "min": float(np.min(cluster_scores)),
            "max": float(np.max(cluster_scores)),
            "negative_count": int(np.sum(cluster_scores < 0)),
        }
    
    return results

def create_silhouette_comparison_plot(
    kmeans_results: dict,
    topic_results: dict,
) -> go.Figure:
    """Create side-by-side silhouette comparison."""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"KMeans Silhouette (Mean: {kmeans_results['mean_score']:.3f})",
            f"Topics Silhouette (Mean: {topic_results['mean_score']:.3f})",
        ),
    )
    
    # KMeans plot
    clusters_km = sorted(kmeans_results["per_cluster"].keys())
    km_means = [kmeans_results["per_cluster"][c]["mean"] for c in clusters_km]
    
    fig.add_trace(
        go.Bar(
            x=[f"C{c}" for c in clusters_km],
            y=km_means,
            marker_color=["green" if s > 0 else "red" for s in km_means],
            name="KMeans",
        ),
        row=1, col=1,
    )
    
    # Topics plot
    clusters_topic = sorted(topic_results["per_cluster"].keys())
    topic_means = [topic_results["per_cluster"][c]["mean"] for c in clusters_topic]
    
    fig.add_trace(
        go.Bar(
            x=[f"T{c}" for c in clusters_topic],
            y=topic_means,
            marker_color=["green" if s > 0 else "red" for s in topic_means],
            name="Topics",
        ),
        row=1, col=2,
    )
    
    fig.update_yaxes(title_text="Silhouette Score", row=1, col=1)
    fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)
    fig.update_layout(height=400, width=1000, showlegend=False)
    
    return fig

# Compute silhouette analysis
if embeddings_dir.exists() and 'kmeans_labels' in globals():
    print("Computing silhouette analysis...")
    
    kmeans_sil = compute_silhouette_analysis(embeddings, kmeans_labels, "KMeans")
    print(f"KMeans Silhouette Score: {kmeans_sil['mean_score']:.4f}")
    
    topic_sil = compute_silhouette_analysis(embeddings, topic_labels, "Topics")
    print(f"Topics Silhouette Score: {topic_sil['mean_score']:.4f}")
    
    # Create comparison visualization
    sil_fig = create_silhouette_comparison_plot(kmeans_sil, topic_sil)
    sil_fig.show()
    
    print("\n" + "="*60)
    print("Silhouette Analysis Summary")
    print("="*60)
    print(f"KMeans Mean: {kmeans_sil['mean_score']:.4f}")
    print(f"Topics Mean: {topic_sil['mean_score']:.4f}")
    print(f"Better clustering: {'KMeans' if kmeans_sil['mean_score'] > topic_sil['mean_score'] else 'Topics'}")

## Section 9: Interactive Search Interface

In [ ]:
# Interactive search interface
if IN_COLAB:
    from ipywidgets import Text, Button, Output, VBox, HBox, HTML
    from IPython.display import clear_output, display
    
    output = Output()
    search_input = Text(
        placeholder='Enter your search query...',
        description='Search:',
        style={'description_width': '80px'},
        layout={'width': '70%'}
    )
    
    def on_search_clicked(b):
        with output:
            clear_output(wait=True)
            query = search_input.value
            
            if not query.strip():
                print("Please enter a search query.")
                return
            
            if embeddings_dir.exists() and 'faiss_index' in globals():
                print(f"Searching for: '{query}'\\n")
                results = search_faiss(
                    query,
                    faiss_index,
                    metadata,
                    embeddings,
                    model_embedding,
                    k=10,
                )
                
                for result in results:
                    print(f"[{result['rank']}] Score: {result['score']:.4f}")
                    print(f"    🔗 {result['title']}")
                    print(f"    📁 {result['category']}")
                    print(f"    📅 {result['date']}")
                    print(f"    📝 {result['snippet']}")
                    print()
            else:
                print("Search index not available. Please load embeddings first.")
    
    search_button = Button(description='Search', button_style='info')
    search_button.on_click(on_search_clicked)
    
    search_box = VBox([
        HTML("<h3>🔍 Semantic Search</h3>"),
        HBox([search_input, search_button]),
        output
    ])
    
    display(search_box)

else:
    print("Interactive search interface available in Google Colab only.")
    print("\nTo test search, use:")
    print("results = search_faiss()")
    print("    'your query here',")
    print("    faiss_index,")
    print("    metadata,")
    print("    embeddings,")
    print("    model_embedding,")
    print("    k=10")
    print(")")

## Summary

This notebook implements a complete semantic search pipeline for the MMC article dataset:

1. **Setup** - Install dependencies and configure Colab environment
2. **Data Loading** - Extract and clean articles from YAML
3. **Embeddings** - Generate semantic embeddings using multilingual E5 model
4. **Storage** - Save embeddings and metadata for reuse
5. **Search Index** - Build FAISS index for fast similarity search
6. **Search** - Execute semantic queries and retrieve similar articles
7. **Visualization** - Project embeddings to 2D using UMAP with KMeans clustering
8. **Quality Analysis** - Evaluate clustering quality with silhouette scores
9. **Interface** - Interactive search widget for exploration

### Key Features:
- ✅ Multilingual embeddings (1024-dim, normalized)
- ✅ Fast cosine similarity search via FAISS
- ✅ 2D visualization with interactive Plotly charts
- ✅ Clustering quality assessment
- ✅ Full Colab integration with GPU support

### To Run in Google Colab:
1. Open this notebook in Colab: `File → Open notebook → Upload`
2. Run cells sequentially from top to bottom
3. Upload your `mmc.yaml` file to Google Drive or use sample data
4. Use the interactive search widget to explore the dataset

### Data Pipeline:
```
YAML → Clean/Extract → Embed → Index → Search/Visualize → Analysis
```

All outputs are saved to `data/mmc_embeddings/` for reproducibility.